In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from tqdm import tqdm
import json
import warnings

warnings.filterwarnings('ignore')

## incs_no 기준 -> sess_id기준으로 데이터 변경 해야함!

In [3]:
# 추후 로그 나눠서 저장해놓고 쓰는걸로 바꾸기!
log_path = './data/LOG.csv'
msg_path = './data/msg_indicate.csv'
cust_path = './data/CUST.csv'
output_path = './data/grp_sess.csv'

In [4]:
log = pd.read_csv(log_path)
print(log.shape)
log.head(2)

(3011584, 100)


,Unnamed: 0,STND_YMD,SITE_ID,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,GOOGLE_CID,SESS_ID,ADID,INCS_NO,...,DVCE_MKT_NM,DVCE_OPRT_NM,DVCE_OPRT_VER,NEW_VISIT_YN,STORE_CD,INVN_ST_YN,SESS_SN_STAY_TIME,EXPS_RSLT_LIST,LIST_TP_NM,ETL_PROC_DTTM
0,0,2024-11-09,UA-110770460-3,APP,a04609a895175b674902dbeba7cadd84384ab60bf52572...,2024-11-09 09:08:38.950,0DB549CA96454588804E5E6CA6E54D88,0DB549CA96454588804E5E6CA6E54D881731110681,1D9F53A0-97B7-409F-BE54-8F94E990D658,0b7163ba5b5a99149416ab8e9dfafedb4fd585be548c6f...,...,NaN,iOS,iOS 17.6.1,N,NaN,Y,0.01,NaN,NaN,2024-11-12 09:38:47.877
1,1,2024-11-09,UA-110770460-3,APP,97bf34fc66ce5f903732c5e8fd3b1f5bbc8bed1dab7a57...,2024-11-09 09:08:53.951,0DB549CA96454588804E5E6CA6E54D88,0DB549CA96454588804E5E6CA6E54D881731110681,1D9F53A0-97B7-409F-BE54-8F94E990D658,0b7163ba5b5a99149416ab8e9dfafedb4fd585be548c6f...,...,NaN,iOS,iOS 17.6.1,N,NaN,Y,0.04,NaN,NaN,2024-11-12 09:38:47.877


In [5]:
use_col_list = ['STND_YMD', 'WEB_APP_CL_CD', 'LOG_SEQ', 'LOG_DTTM', 'SESS_ID', 'INCS_NO', 'AGE', 'BRTH_YEAR', 
                'SEX_CD', 'EMP_YN', 'CUST_GRD_NM', 'DVCE_TP_CD', 'SITE_URL', 'PG_URL', 'PG_NM', 'PG_LOC_VL', 
                'PG_TP_VL', 'SVC_CL_CD', 'UTM_SOURCE', 'ACCM_STAY_TIME', 'PG_STAY_TIME', 'EVNT_NM', 'EVNT_CAT_DTL',
                'ORD_AMT', 'CPN_DC_AMT', 'MBL_GFCR_DC_AMT', 'BTPN_DC_AMT', 'GIFT_CD_AMT', 'PRD_INFO', 'DVCE_MDL_NM']

log = log[use_col_list]
log.head(2)

,STND_YMD,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,SESS_ID,INCS_NO,AGE,BRTH_YEAR,SEX_CD,EMP_YN,...,PG_STAY_TIME,EVNT_NM,EVNT_CAT_DTL,ORD_AMT,CPN_DC_AMT,MBL_GFCR_DC_AMT,BTPN_DC_AMT,GIFT_CD_AMT,PRD_INFO,DVCE_MDL_NM
0,2024-11-09,APP,a04609a895175b674902dbeba7cadd84384ab60bf52572...,2024-11-09 09:08:38.950,0DB549CA96454588804E5E6CA6E54D881731110681,0b7163ba5b5a99149416ab8e9dfafedb4fd585be548c6f...,50.0,1974,F,N,...,NaN,view_item,ecommerce,NaN,0.0,0.0,0.0,0.0,"[\n {\n ""ad_nm"": ""(not set)"",\n ""ad_slo...",iPhone 14 Pro
1,2024-11-09,APP,97bf34fc66ce5f903732c5e8fd3b1f5bbc8bed1dab7a57...,2024-11-09 09:08:53.951,0DB549CA96454588804E5E6CA6E54D881731110681,0b7163ba5b5a99149416ab8e9dfafedb4fd585be548c6f...,50.0,1974,F,N,...,NaN,view_item,ecommerce,NaN,0.0,0.0,0.0,0.0,"[\n {\n ""ad_nm"": ""(not set)"",\n ""ad_slo...",iPhone 14 Pro


### PRD_INFO 파싱

In [6]:
# JSON 형식의 문자열 딕셔너리로 변환하는 함수 정의
def json_str_to_dict(json_str):
    if isinstance(json_str, float) and np.isnan(json_str):
        return {} # 빈 딕셔너리 반환
    pattern = re.compile(r'\"(.*?)\":\s*\"(.*?)\",*\n*')
    matches = pattern.findall(json_str)
    result_dict = {key: (value if value != '(not set)' else None) for key, value in matches}
    return result_dict

# PRD_INFO 컬럼에 함수를 적용하여 파싱된 데이터를 새로운 칼럼으로 추가하는 함수 정의
def apply_and_expand(df, col_name):
    # 각 행의 데이터를 딕셔너리로 변환
    df_dicts = df[col_name].apply(json_str_to_dict)
    
    # 딕셔너리 형태의 데이터를 데이터프레임으로 변환
    expanded_df = pd.json_normalize(df_dicts)
    
    # 생성된 새로운 데이터프레임을 기존 데이터프레임에 병합
    result_df = pd.concat([df, expanded_df], axis=1)
    
    return result_df


# apply_and_expand 함수를 사용하여 데이터프레임을 업데이트
log_prse = apply_and_expand(log, 'PRD_INFO')
print(log_prse.shape)
log_prse.head(2)

(3011584, 51)


,STND_YMD,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,SESS_ID,INCS_NO,AGE,BRTH_YEAR,SEX_CD,EMP_YN,...,prd_loc,prd_norm_prc,prd_optn,prd_prom_id,prd_prom_nm,prd_qty,prd_sal_prc,prd_tp_cat_vl,acml_bt_pt,prd_sal_amt
0,2024-11-09,APP,a04609a895175b674902dbeba7cadd84384ab60bf52572...,2024-11-09 09:08:38.950,0DB549CA96454588804E5E6CA6E54D881731110681,0b7163ba5b5a99149416ab8e9dfafedb4fd585be548c6f...,50.0,1974,F,N,...,None,8000,None,None,None,1,1000,스킨케어,NaN,NaN
1,2024-11-09,APP,97bf34fc66ce5f903732c5e8fd3b1f5bbc8bed1dab7a57...,2024-11-09 09:08:53.951,0DB549CA96454588804E5E6CA6E54D881731110681,0b7163ba5b5a99149416ab8e9dfafedb4fd585be548c6f...,50.0,1974,F,N,...,None,8000,None,None,None,1,1000,스킨케어,NaN,NaN


### 로그데이터 그래프 임베딩
* 각 페이지에 상응하는 임베딩 벡터값 필요!

In [7]:
log_prse['PG_EMB_VCT'] = 0

### 로그데이터 확인

In [8]:
grp_sess = pd.DataFrame(log_prse.groupby('SESS_ID'))
grp_sess.columns = ['SESS_ID', 'LOG']
print(grp_sess.shape)
grp_sess.head(2)

(258405, 2)


,SESS_ID,LOG
0,0005ad80698368c16258c365c05c76a21732151843,STND_YMD WEB_APP_CL_CD \ 9291 2024-1...
1,0005ad80698368c16258c365c05c76a21732276772,STND_YMD WEB_APP_CL_CD \ 9296 2024-1...


### 로그 전처리

* 날짜 간격 2일이상 -> 다른 세션으로 분리 / sess_Start를 기준으로 세션 분리
* 포인트 관련 데이터 삭제
* 앱여부 0,1

In [9]:
# 데이터 코딩 / 오래걸림 엄청 다시 안돌리게 주의..
len_grp_sess = len(grp_sess)

for i in tqdm(range(len_grp_sess)):
    grp_sess['LOG'][i] = grp_sess['LOG'][i] 
    grp_sess['LOG'][i]['LOG_DTTM'] = pd.to_datetime(grp_sess['LOG'][i]['LOG_DTTM'])
    grp_sess['LOG'][i] = grp_sess['LOG'][i].sort_values(by='LOG_DTTM')
    grp_sess['LOG'][i] = grp_sess['LOG'][i].reset_index(drop=True)
    
    if (grp_sess['LOG'][i]['WEB_APP_CL_CD'] == 'APP').any():
        grp_sess['LOG'][i]['APP'] = 1
    else:
        grp_sess['LOG'][i]['APP'] = 0

100%|██████████| 258405/258405 [7:42:32<00:00,  9.31it/s]       


In [16]:
temp = grp_sess['LOG'][0]
temp.iloc[0]['INCS_NO']

'edda90412258df2570f4e54f2ad5b79b611b30438594bbc0e8c328658bfdff0b7cbfe13dadf02f97168dc23afb0b73443242022d2d91a8a73e99109b4a56861a'

In [ ]:
# night 추가, incs 수정

In [ ]:
# 데이터 인디케이팅 -> incs부분 수정
SESS_TIME_list = []
SRCH_EFRT_list = []
PRDV_CNT_list = []
CAT_CNT_list = []
SAME_PAGE_CNT_list = []
EVNT_CNT_list = []
incs_no = []
lst_sess_time = []
night_list = []

for i in tqdm(range(len_grp_sess)):
    temp = grp_sess['LOG'][i]

    # 전체 세션 시간(time) -> 안쓰는게 나을듯
    SESS_TIME_list.append(np.sum(temp['PG_STAY_TIME']))

    # 탐색 노력(전체 세션 개수)
    SRCH_EFRT_list.append(len(temp))

    # 상품 뷰 숫자
    PRDV_CNT_list.append((len(temp[temp['PRD_INFO'].notna()])))

    # 카테고리 뷰 숫자
    CAT_CNT_list.append(len(set(temp[temp['PRD_INFO'].notna()]['prd_tp_cat_vl'])) - 1)

    # 동일한 페이지를 본 숫자
    SAME_PAGE_CNT_list.append(len(set(temp['PG_URL'])))

    # 이벤트 페이지 탐색 횟수 확인
    EVNT_CNT_list.append(len(temp[temp['EVNT_NM'].notna()]))

    incs_no.append(temp.iloc[0]['INCS_NO'])    
    lst_sess_time.append(temp.iloc[-1]['LOG_DTTM'])

  0%|          | 0/258405 [00:00<?, ?it/s]

100%|██████████| 258405/258405 [08:33<00:00, 503.10it/s]


In [21]:
sess_indicate = pd.DataFrame({'SESS_ID' : grp_sess['SESS_ID'],
        # 'SESS_TIME':SESS_TIME_list,
        'SRCH_EFRT' : SRCH_EFRT_list,
        'PRDV_CNT' : PRDV_CNT_list,
        'CAT_CNT' : CAT_CNT_list,
        'SAMGE_PAGE_CNT' : SAME_PAGE_CNT_list,
        'EVNT_CNT' : EVNT_CNT_list,
        'INCS_NO' : incs_no,
        'LST_SESS_TIME' : lst_sess_time
        })

print(sess_indicate.shape)
sess_indicate.head(2)

(258405, 8)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME
0,0005ad80698368c16258c365c05c76a21732151843,7,1,0,5,7,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-21 10:19:01.273
1,0005ad80698368c16258c365c05c76a21732276772,4,1,0,2,4,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-22 20:59:44.030


### SCROLL depth 추가

In [26]:
scroll = pd.read_csv('./raw/SCROLL.csv')
print(scroll.shape)
scroll.head(2)

(7034670, 3)


,SESS_ID,INCS_NO,EVNT_ACTN
0,a175d88e4c2e493fefd2f85a8d36811f1730531578,2b917dbf9f2402193ef72a75bdee82d40239140096e82e...,10%
1,a175d88e4c2e493fefd2f85a8d36811f1730531578,2b917dbf9f2402193ef72a75bdee82d40239140096e82e...,10%


In [29]:
def del_per(text):
    text = int(text.split('%')[0])
    return text

In [30]:
scroll['EVNT_ACTN'] = scroll['EVNT_ACTN'].apply(del_per)
scroll.head(2)

,SESS_ID,INCS_NO,EVNT_ACTN
0,a175d88e4c2e493fefd2f85a8d36811f1730531578,2b917dbf9f2402193ef72a75bdee82d40239140096e82e...,10
1,a175d88e4c2e493fefd2f85a8d36811f1730531578,2b917dbf9f2402193ef72a75bdee82d40239140096e82e...,10


In [31]:
scroll_grp = scroll.groupby('SESS_ID').agg(AVG_DEPTH=('EVNT_ACTN', 'mean')).reset_index()
print(scroll_grp.shape)
scroll_grp.head(2)

(410246, 2)


,SESS_ID,AVG_DEPTH
0,0001ea288f4588344483894291db12a61731219591,35.428571
1,0001ea288f4588344483894291db12a61731230728,35.714286


In [32]:
sess_indicate = pd.merge(sess_indicate, scroll_grp, on='SESS_ID', how='left')

# AVG_DEPTH가 없는 값은 0으로 채우기
sess_indicate['AVG_DEPTH'] = sess_indicate['AVG_DEPTH'].fillna(0)
sess_indicate.head(2)

,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH
0,0005ad80698368c16258c365c05c76a21732151843,7,1,0,5,7,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-21 10:19:01.273,50.909091
1,0005ad80698368c16258c365c05c76a21732276772,4,1,0,2,4,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-22 20:59:44.030,25.000000


### day_off 추가

In [44]:
def day_off_check(dt):
    dt = dt.time()
    evening_start = pd.Timestamp('18:00:00').time()
    morning_end = pd.Timestamp('09:00:00').time()
    
    # 저녁 6시 이후 또는 오전 9시 이전인지 확인
    if dt >= evening_start or dt < morning_end:
        return 1
    else:
        return 0

In [45]:
sess_indicate['day_off'] = sess_indicate['LST_SESS_TIME'].apply(day_off_check)
sess_indicate.head(2)

,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off
0,0005ad80698368c16258c365c05c76a21732151843,7,1,0,5,7,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-21 10:19:01.273,50.909091,0
1,0005ad80698368c16258c365c05c76a21732276772,4,1,0,2,4,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-22 20:59:44.030,25.000000,1


In [51]:
# 탐색 노력이 7이상인 데이터만 사용 -> 추후 탐색노력에 대한 영향이 없다는 추가 분석 필요
sess_indicate_cutoff = sess_indicate[sess_indicate["SRCH_EFRT"] > 6]
sess_indicate_cutoff = sess_indicate_cutoff[sess_indicate_cutoff['AVG_DEPTH']!=0]

sess_indicate.to_csv('./data/sess_indicate_origin.csv')
sess_indicate_cutoff.to_csv('./data/sess_indicate_cutoff7.csv')

In [50]:
print(sess_indicate_cutoff.shape)
sess_indicate_cutoff.head(2)

(82294, 10)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off
0,0005ad80698368c16258c365c05c76a21732151843,7,1,0,5,7,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-21 10:19:01.273,50.909091,0
3,000A204228DB4108A69885BF2BCA220E1730848747,70,13,1,23,70,51dea90cd3bb8eb26f2580e7891ebb2d38e1d820b8b85a...,2024-11-06 08:39:48.598,33.965517,1
